In [ ]:
# ── Settings ── change only these ────────────────────────────────────────────
TRAIN_HOURS = 4       # 0.25 = pilot. Set to 4 for the full training
MAX_CONTEXT = 12288      # longest example in tokens: longer ones cost too much time
MAX_TARGET = 1536        # longest edit call in tokens
RANK = 16
LAYERS = list(range(30, 60))                     # only the later layers train, which saves time
TARGETS = {"q_proj": "self_attn", "o_proj": "self_attn", "down_proj": "mlp"}
LEARNING_RATE = 2e-4
ACCUMULATE = 4           # examples per weight update
SAVE_EVERY_MINUTES = 30
HELD_OUT = 8             # examples from other repositories, for the before and after loss
SEED = 42
ATTN_CHUNK = 2048

import os
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
for key, value in {"TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_FLAX": "1", "TRANSFORMERS_OFFLINE": "1",
                   "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}.items():
    os.environ[key] = value


In [ ]:
# ── Check everything first, then install the pinned versions (no internet needed) ──
import importlib.util
import subprocess
import sys

PINNED = ("transformers-", "tokenizers-", "safetensors-")
paths = [p for p in INPUT.rglob("*") if p.is_file()]


def wheel_for(prefix):
    found = [p for p in paths if p.suffix == ".whl" and p.name.startswith(prefix)]
    return sorted(found, key=lambda p: "wheelhouse" not in str(p))[0] if found else None


wheels = {prefix: wheel_for(prefix) for prefix in PINNED}
bnb_wheels = sorted({p for p in paths if p.suffix == ".whl" and p.name.startswith("bitsandbytes-")}, key=lambda p: "wheelhouse" not in str(p))
try:
    gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ")
except FileNotFoundError:
    gpus = 0

problems = []
if not all(wheels.values()):
    problems.append("Add Input, dataset: metric/gemma-4-developer-agent-wheelhouse (no " + ", ".join(k for k, v in wheels.items() if not v) + " wheel)")
if not any(p.name == "lora_index.csv" for p in paths):
    problems.append("Add Input, dataset: dmitriigluzdov/gemma-4-serving-and-lora-verification-kit")
if not any(p.name == "config.json" and "q4_0-unquantized" in str(p) for p in paths):
    problems.append("Add Input, model: google/gemma-4, variation gemma-4-31b-it-qat-q4_0-unquantized")
if gpus != 4:
    problems.append(f"Settings, Accelerator: GPU L4 x4 (this session has {gpus} GPUs)")
if importlib.util.find_spec("accelerate") is None:
    problems.append("The accelerate package is missing from this Kaggle environment")

print("Attached inputs:")
for folder in sorted({"/".join(p.relative_to(INPUT).parts[:3]) for p in paths}):
    print("  ", folder)
assert not problems, "Fix these, then run again:\n- " + "\n- ".join(problems)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *map(str, wheels.values())], check=True)
print("installed", len(wheels), "pinned wheels on", gpus, "GPUs")

# ── Does 4-bit work here? Test it on the GPU now, in seconds, instead of finding out after 11 minutes ──
BNB_TEST = """
import torch, bitsandbytes as bnb
x = torch.randn(256, 256, dtype=torch.bfloat16, device="cuda:0")
q, s = bnb.functional.quantize_4bit(x, quant_type="nf4", blocksize=64)
y = bnb.functional.dequantize_4bit(q, s)
print("bitsandbytes", bnb.__version__, "ok", float((x.float() - y.float()).abs().mean()))
"""


def bnb_works(env=None):
    r = subprocess.run([sys.executable, "-c", BNB_TEST], capture_output=True, text=True, env={**os.environ, **(env or {})})
    ok = r.returncode == 0 and " ok " in r.stdout
    return ok, (r.stdout.strip().splitlines() or [""])[-1] if ok else (r.stderr.strip().splitlines() or ["no output"])[-1][:200]


USE_4BIT = False
ok, note = bnb_works()
print("pre-installed bitsandbytes:", "works" if ok else "does not work", "|", note)
USE_4BIT = ok
for wheel in ([] if ok else bnb_wheels):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", str(wheel)], check=True)
    ok, note = bnb_works()
    print(f"bitsandbytes wheel {wheel.name}:", "works" if ok else "does not work", "|", note)
    if ok:
        USE_4BIT = True
        break

if USE_4BIT:
    print("4-bit loading: ON")
else:
    MAX_CONTEXT = min(MAX_CONTEXT, 8192)
    print("4-bit loading: OFF. The model will load in bf16 (about 62 GB over the 4 GPUs); MAX_CONTEXT is now", MAX_CONTEXT)


In [ ]:
# ── Load the model (4-bit if it works here, otherwise bf16; 10 to 30 minutes) ──
import time

import torch
from transformers import AttentionInterface, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.integrations.sdpa_attention import sdpa_attention_forward
from transformers.masking_utils import AttentionMaskInterface, sdpa_mask


def chunked_sdpa(module, query, key, value, attention_mask, dropout=0.0, scaling=None, is_causal=None, **kw):
    """The global-attention layers would need about 16 GiB at 16k tokens. Working in chunks keeps it small with the same result."""
    q_len, kv_len = query.shape[2], key.shape[2]
    if q_len <= ATTN_CHUNK:
        return sdpa_attention_forward(module, query, key, value, attention_mask, dropout=dropout, scaling=scaling, is_causal=is_causal, **kw)
    outs = []
    for s in range(0, q_len, ATTN_CHUNK):
        e = min(q_len, s + ATTN_CHUNK)
        if attention_mask is not None:
            m = attention_mask[:, :, s:e, :]
        else:
            qi = torch.arange(s, e, device=query.device)[:, None] + (kv_len - q_len)
            m = (torch.arange(kv_len, device=query.device)[None, :] <= qi)[None, None]

        def f(q, k, v, mm):
            return sdpa_attention_forward(module, q, k, v, mm, dropout=dropout, scaling=scaling, is_causal=False, **kw)[0]

        if torch.is_grad_enabled():
            outs.append(torch.utils.checkpoint.checkpoint(f, query[:, :, s:e], key, value, m, use_reentrant=False))
        else:
            outs.append(f(query[:, :, s:e], key, value, m))
    return torch.cat(outs, dim=1), None


AttentionInterface.register("chunked_sdpa", chunked_sdpa)
AttentionMaskInterface.register("chunked_sdpa", sdpa_mask)

found = [p.parent for p in INPUT.rglob("config.json") if "q4_0-unquantized" in str(p)]
assert found, "Add the model google/gemma-4, variation gemma-4-31b-it-qat-q4_0-unquantized, with Add Input"
MODEL_DIR = found[0]
tok = AutoTokenizer.from_pretrained(str(MODEL_DIR))
started = time.time()


def load_model(four_bit):
    kwargs = dict(device_map="balanced", max_memory={i: ("20GiB" if four_bit else "19GiB") for i in range(torch.cuda.device_count())},
                  dtype=torch.bfloat16, attn_implementation="chunked_sdpa")
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
            llm_int8_skip_modules=["vision_tower", "audio_tower", "embed_vision", "embed_audio", "lm_head", "multi_modal_projector"])
    return AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), **kwargs)


try:
    model = load_model(USE_4BIT)
except Exception as error:
    if not USE_4BIT:
        raise
    print("4-bit load failed:", repr(error)[:300], "\nLoading in bf16 instead.")
    import gc
    model = None
    gc.collect()
    torch.cuda.empty_cache()
    USE_4BIT = False
    MAX_CONTEXT = min(MAX_CONTEXT, 8192)
    model = load_model(False)
print(f"model loaded in {(time.time() - started) / 60:.1f} minutes on {torch.cuda.device_count()} GPUs")
AUTOCAST_DEVICE = "cuda"


In [ ]:
# ── Choose the training examples: edit_file calls on source files from resolved runs (about 1 minute) ──
import json
import random
import re
from array import array

DATA = next(p.parent for p in INPUT.rglob("lora_index.csv"))
FILES = ["lora_native_resolved.jsonl", "lora_external_A.jsonl"] + [f"lora_external_B.part{i:03d}.jsonl" for i in range(4)]
EDIT_CALL = re.compile(r"<\|tool_call>call:edit_file\{")
PATH = re.compile(r'filepath:<\|"\|>(.*?)<\|"\|>')
NOT_SOURCE = re.compile(r"(^|/)(tests?|testing)/|(^|/)test_[^/]*$|_test\.py$|^/?tmp/|conftest\.py$|(^|/)(repro|reproduce|verify|debug|check|scratch|demo)[^/]*$")
TOOL_CALL_ID = tok.convert_tokens_to_ids("<|tool_call>")


def read_examples():
    """Every kept example holds all its tokens and the position where the edit call starts."""
    examples, skipped = [], {"too long": 0, "not an edit of a source file": 0}
    for name in FILES:
        if not (DATA / name).is_file():
            print("missing", name)
            continue
        with open(DATA / name, encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                repo = record.get("repository") or record.get("repo") or ""
                for ex in record["sft_examples"]:
                    text = ex["completion"]
                    path = PATH.search(text)
                    if text.count("<|tool_call>") != 1 or not EDIT_CALL.search(text) or not path or NOT_SOURCE.search(path[1]):
                        skipped["not an edit of a source file"] += 1
                        continue
                    target = ex.get("completion_token_ids")
                    if not target or TOOL_CALL_ID not in target or ex.get("context_tokens", 0) > MAX_CONTEXT or len(target) > MAX_TARGET:
                        skipped["too long"] += 1
                        continue
                    prompt = ex.get("prompt_token_ids") or tok(ex["prefix"], add_special_tokens=False)["input_ids"]
                    examples.append({"ids": array("I", list(prompt) + list(target)), "start": len(prompt) + target.index(TOOL_CALL_ID),
                                     "repo": repo, "native": "native" in name})
    return examples, skipped


examples, skipped = read_examples()
assert examples, "no examples: add the dataset dmitriigluzdov/gemma-4-serving-and-lora-verification-kit"
rng = random.Random(SEED)
rng.shuffle(examples)

# Hold out whole repositories, then alternate Gemma's own runs and the stronger model's runs
repos = sorted({e["repo"] for e in examples})
rng.shuffle(repos)
held, held_repos = [], set()
for repo in repos:
    if len(held) >= HELD_OUT:
        break
    held_repos.add(repo)
    held += [e for e in examples if e["repo"] == repo]
held = held[:HELD_OUT]
pool = [e for e in examples if e["repo"] not in held_repos]
native = [e for e in pool if e["native"]]
external = [e for e in pool if not e["native"]]
train = []
for i in range(max(len(native), len(external))):
    train += [group[i] for group in (native, external) if i < len(group)]

tokens = sum(len(e["ids"]) for e in train)
print(f"train: {len(train)} examples ({len(native)} Gemma, {len(external)} Qwen), {tokens:,} tokens; held out: {len(held)}; skipped: {skipped}")
print(f"supervised tokens (only the edit calls): {sum(len(e['ids']) - e['start'] for e in train):,}")
assert len(train) >= 20 and len(held) >= 1


In [ ]:
# ── Add the LoRA by hand, so nothing else in the model can train ──
import math

from torch import nn


class LoraLinear(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        device = next(base.parameters()).device
        self.lora_A = nn.Linear(base.in_features, RANK, bias=False, device=device, dtype=torch.float32)
        self.lora_B = nn.Linear(RANK, base.out_features, bias=False, device=device, dtype=torch.float32)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x, *args, **kwargs):
        return self.base(x, *args, **kwargs) + self.lora_B(self.lora_A(x)).to(x.dtype)


for p in model.parameters():
    p.requires_grad_(False)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
WRAP = re.compile(r".*language_model\.layers\.(\d+)\.(self_attn|mlp)\.(\w+_proj)$")
lora = {}
for name, module in list(model.named_modules()):
    m = WRAP.fullmatch(name)
    if m and int(m[1]) in LAYERS and TARGETS.get(m[3]) == m[2]:
        wrapped = LoraLinear(module)
        setattr(model.get_submodule(name.rsplit(".", 1)[0]), m[3], wrapped)
        lora[f"layers.{m[1]}.{m[2]}.{m[3]}"] = wrapped
assert len(lora) == len(LAYERS) * len(TARGETS), f"wrapped {len(lora)} modules"
params = [p for w in lora.values() for p in (w.lora_A.weight, w.lora_B.weight)]
print(f"LoRA on {len(lora)} modules, {sum(p.numel() for p in params):,} trainable values")


In [ ]:
# ── Train only on the edit calls, and stop when the time is up ──
from safetensors.torch import save_file

SOFTCAP = getattr(model.config.get_text_config(), "final_logit_softcapping", None)
lm_head = model.get_output_embeddings()
input_device = model.get_input_embeddings().weight.device
OUT = WORK / "trained_adapter" / "main_lora"


def example_loss(example):
    ids = torch.tensor(list(example["ids"]), device=input_device)
    with torch.autocast(AUTOCAST_DEVICE, dtype=torch.bfloat16):
        hidden = model.model(input_ids=ids[None]).last_hidden_state[0]
    positions = torch.arange(example["start"] - 1, len(ids) - 1, device=hidden.device)
    logits = lm_head(hidden[positions].to(lm_head.weight.device)).float()
    if SOFTCAP:
        logits = torch.tanh(logits / SOFTCAP) * SOFTCAP
    # the ids live on one GPU and the hidden states on another, so move the index to the ids' GPU first
    targets = ids[(positions + 1).to(ids.device)].to(logits.device)
    return torch.nn.functional.cross_entropy(logits, targets)


@torch.no_grad()
def held_out_loss():
    return sum(example_loss(e).item() for e in held) / len(held)


def save_adapter():
    OUT.mkdir(parents=True, exist_ok=True)
    tensors = {}
    for name, wrapped in lora.items():
        prefix = f"base_model.model.model.language_model.{name}"
        tensors[f"{prefix}.lora_A.weight"] = wrapped.lora_A.weight.detach().to(torch.bfloat16).cpu().contiguous()
        tensors[f"{prefix}.lora_B.weight"] = wrapped.lora_B.weight.detach().to(torch.bfloat16).cpu().contiguous()
    save_file(tensors, str(OUT / "adapter_model.safetensors"), metadata={"format": "pt"})
    (OUT / "adapter_config.json").write_text(json.dumps({
        "peft_type": "LORA", "task_type": "CAUSAL_LM", "base_model_name_or_path": "google/gemma-4-31b-it-qat-w4a16-ct",
        "r": RANK, "lora_alpha": RANK, "lora_dropout": 0.0, "bias": "none", "fan_in_fan_out": False, "inference_mode": True,
        "init_lora_weights": True, "use_dora": False, "use_rslora": False, "modules_to_save": None,
        "target_modules": sorted(TARGETS), "layers_to_transform": LAYERS, "layers_pattern": None,
        "rank_pattern": {}, "alpha_pattern": {}, "revision": None}, indent=1))
    return len(tensors)


model.train()
before = held_out_loss()
print(f"held-out loss before training: {before:.4f}")
save_adapter()

optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=0.0)
limit = TRAIN_HOURS * 3600
started = last_save = time.time()
seen = tokens_done = updates = out_of_memory = 0
running = []
for example in (train[i % len(train)] for i in range(10 ** 9)):
    elapsed = time.time() - started
    if elapsed >= limit:
        break
    done = elapsed / limit
    for group in optimizer.param_groups:
        group["lr"] = LEARNING_RATE * min(1.0, done / 0.05 + 0.05) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * done)))
    try:
        loss = example_loss(example)
        (loss / ACCUMULATE).backward()
    except torch.cuda.OutOfMemoryError:
        out_of_memory += 1
        optimizer.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        if out_of_memory >= 25 and seen == 0:
            raise RuntimeError("The first 25 examples all ran out of memory. Lower MAX_CONTEXT (for example to 6144) and run again.")
        continue
    assert torch.isfinite(loss), "the loss is not a number"
    seen += 1
    tokens_done += len(example["ids"])
    running.append(loss.item())
    if seen % ACCUMULATE == 0:
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        updates += 1
        print(f"{elapsed / 60:6.1f} min | update {updates} | {seen} examples | loss {sum(running) / len(running):.4f} | "
              f"{1000 * elapsed / tokens_done:.1f} ms per token", flush=True)
        running = []
    if time.time() - last_save >= SAVE_EVERY_MINUTES * 60:
        save_adapter()
        last_save = time.time()

saved = save_adapter()
after = held_out_loss()
result = {"examples_seen": seen, "of": len(train), "updates": updates, "out_of_memory_skips": out_of_memory, "tokens": tokens_done,
          "ms_per_token": round(1000 * (time.time() - started) / max(tokens_done, 1), 1),
          "held_out_loss_before": round(before, 4), "held_out_loss_after": round(after, 4)}
(WORK / "trained_adapter" / "result.json").write_text(json.dumps(result, indent=1))
print(json.dumps(result, indent=1))
assert updates >= 1 and (OUT / "adapter_model.safetensors").is_file()
print(f"OK: adapter saved, {saved} tensors, {(OUT / 'adapter_model.safetensors').stat().st_size / 2 ** 20:.0f} MB")